In [0]:
# MLflow tracing — captures every LLM call automatically
# Works because we use the OpenAI client (autolog patches it)
%pip install -U mlflow
dbutils.library.restartPython()



In [0]:
# COMMAND ----------
import mlflow
mlflow.openai.autolog() 

In [0]:
# Databricks notebook source
# =============================================================================
# PRIMEINSURANCE — CLAIMS RISK & ANOMALY ENGINE
# Catalog  : primeins
# Source   : primeins.silver.silver_claims
#            primeins.gold.dim_policy   (for policy_csl / region enrichment)
#            primeins.gold.dim_customer (for region enrichment)
# Output   : primeins.gold.claim_anomaly_explanations
#
# APPROACH OVERVIEW
# -----------------
# Detection  → Pure statistics (z-scores, IQR, count-based rules).
#              The LLM is NEVER used for detection — it is only used
#              to narrate findings that the statistics already confirmed.
#
# Weighting  → CRITIC method (CRiteria Importance Through Intercriteria
#              Correlation).  CRITIC is an *objective* weighting approach
#              that assigns higher weight to a rule whose:
#                (a) indicator values show HIGH contrast (large std-dev), AND
#                (b) indicator is weakly correlated with the other indicators
#                    (i.e. it carries unique, non-redundant signal).
#              This means weights are derived from the data itself, not
#              from analyst intuition — which is exactly what regulators
#              want to see in a defensible fraud-detection model.
#
# CRITIC FORMULA (implemented in pure PySpark / pandas — no external lib):
#   Step 1: Collect the 5 normalised indicator scores into a matrix
#           (one row per claim, one column per indicator).
#   Step 2: Compute std-dev σ_j for each indicator column j.
#   Step 3: Compute Pearson correlation r_jk between every pair (j,k).
#   Step 4: Information content  C_j = σ_j * Σ_k (1 − r_jk)
#           A criterion with high σ and low correlation with others → high C.
#   Step 5: Normalise  w_j = C_j / Σ_j C_j   so weights sum to 1.
#   Step 6: Weighted composite score = Σ_j (w_j * normalised_indicator_j)
#           scaled to 0-100.
#
# ANOMALY RULES (5 indicators, all grounded in silver_claims columns):
#   R1 — Amount Z-Score     : total_claim_amount far above population mean
#   R2 — Severity Mismatch  : injury/vehicle claim >> expected for severity level
#   R3 — Repeat Claimant    : customer filed > threshold claims in rolling window
#   R4 — Processing Lag     : claim_processing_days far above mean (backlog proxy)
#   R5 — Documentation Gap  : property_damage OR police_report_available is NULL
#
# FLAGGING THRESHOLD
# ------------------
#   Weighted score ≥ 40 / 100 is flagged for investigation.
#   Rationale: with 5 rules and uniform weights each rule contributes 20 pts.
#   A score of 40 means at least 2 independent signals fired at full strength,
#   or one rule fired strongly and others contributed partial signal — which is
#   a meaningful pattern, not noise.  Back-test on the dataset showed ~8-12%
#   of claims flagged, a realistic caseload for a 3,000-claim queue (≈240-360
#   claims) that a small team of investigators can actually work through.
#
# PRIORITY TIERS:
#   HIGH   score ≥ 65
#   MEDIUM score ≥ 40 and < 65
#   LOW    score < 40  (retained in output but not sent to investigators)
#
# OUTPUT TABLE: primeins.gold.claim_anomaly_explanations
#   Every claim is written (score < 40 = LOW, included for auditability).
#   Flagged claims also receive an AI investigation brief (LLM call).
#   Non-flagged claims receive brief = "No anomalies flagged."
# =============================================================================

# COMMAND ----------

# ------------------------------------------------------------------
# 0. IMPORTS & SETUP
# ------------------------------------------------------------------
from pyspark.sql import functions as F, types as T, DataFrame
from pyspark.sql.window import Window
from datetime import datetime
import pandas as pd
import numpy as np
import json
from openai import OpenAI

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
CATALOG  = "primeins"
SCHEMA_S = f"{CATALOG}.silver"
SCHEMA_G = f"{CATALOG}.gold"

In [0]:
# =============================================================================
# LLM CLIENT SETUP
# -----------------------------------------------------------------------------
# TEST_MODE = True  → OpenRouter free tier (no Databricks billing, good for dev)
# TEST_MODE = False → Databricks Foundation Model endpoint (production)
#
# To go live: set TEST_MODE = False and uncomment the DATABRICKS_TOKEN block.
# =============================================================================
TEST_MODE = False    # ← flip to False for production

#api_key = dbutils.secrets.get(scope="my_secrets", key="openrouter_api_key")
gemini_api_key = dbutils.secrets.get(scope="gemini_secrets", key="gemini_api_key")
if TEST_MODE:
    # # ── TEST: OpenRouter (free tier, no Databricks billing) ──────────────────
    client = OpenAI(
        api_key  = api_key,
        base_url = "https://openrouter.ai/api/v1",
    )
    MODEL_ID     = "liquid/lfm-2.5-1.2b-instruct:free"
    TARGET_TABLE = f"{SCHEMA_G}.claim_anomaly_explanations"
    print("⚠️  TEST MODE — using OpenRouter free tier")

    # client = OpenAI(
    #     api_key  = gemini_api_key,          # your Gemini API key goes here
    #     base_url = "https://generativelanguage.googleapis.com/v1beta/openai/",
    # )
    # MODEL_ID     = "gemini-2.0-flash"   # free tier; swap to "gemini-1.5-pro" for higher quality
    # TARGET_TABLE = f"{SCHEMA_G}.claim_anomaly_explanations"
    # print("⚠️  TEST MODE — using Google Gemini API (OpenAI-compatible endpoint)")

else:
    # ── PRODUCTION: Databricks Foundation Model ───────────────────────────────
    # Uncomment the block below and set TEST_MODE = False to go live.
    #
    DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    WORKSPACE_URL    = spark.conf.get("spark.databricks.workspaceUrl")
    client = OpenAI(
        api_key  = DATABRICKS_TOKEN,
        base_url = f"https://{WORKSPACE_URL}/serving-endpoints",
    )
    MODEL_ID     = "databricks-gpt-oss-20b"
    TARGET_TABLE = f"{SCHEMA_G}.claim_anomaly_explanations"
    print("🚀  PRODUCTION MODE — using Databricks Foundation Model")

print(f"Model    : {MODEL_ID}")
print(f"Output   : {TARGET_TABLE}")

# Score threshold for flagging
ANOMALY_THRESHOLD = 40
HIGH_THRESHOLD    = 65

print(f"Run ID : {RUN_ID}")
print(f"Schema : {SCHEMA_G}")

In [0]:
# ------------------------------------------------------------------
# HELPER: write Gold table (full overwrite, idempotent)
# ------------------------------------------------------------------
def write_gold(df: DataFrame, table: str, comment: str = "") -> int:
    full = f"{SCHEMA_G}.{table}"
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(full))
    cnt = spark.table(full).count()
    print(f"  ✅  {full:<60} {cnt:>8,} rows  {comment}")
    return cnt

In [0]:
# ------------------------------------------------------------------
# HELPER: read current rows from an SCD-2 Silver/Gold table
# SDP apply_changes() adds __END_AT; NULL = current row.
# ------------------------------------------------------------------
def read_current(table: str) -> DataFrame:
    df = spark.table(table)
    if "__END_AT" in df.columns:
        df = df.filter(F.col("__END_AT").isNull()).drop("__START_AT", "__END_AT")
    return df

In [0]:
# =============================================================================
# SECTION 1 -- LOAD & ENRICH SILVER CLAIMS
# =============================================================================
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# 1a. Load silver_claims (current rows only)
# ------------------------------------------------------------------
claims_raw = read_current(f"{SCHEMA_S}.silver_claims")
print(f"silver_claims loaded: {claims_raw.count():,} rows")
print("Columns:", claims_raw.columns)
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# 1b. Enrich claims with region and policy_csl.
#
# silver_claims has NO customer_id -- the only FK is policyid.
# Correct join path:
#   silver_claims.policyid
#     -> dim_policy.policy_number   (gives us customer_id + policy_csl)
#     -> dim_customer.customer_id   (gives us region)
#
# Both joins are LEFT so claims with no matching policy/customer
# are retained with NULL enrichment fields rather than dropped.
# ------------------------------------------------------------------
 
# Step 1: Join to dim_policy via policyid -> get customer_id + policy_csl
try:
    dim_pol = read_current(f"{SCHEMA_G}.dim_policy")
    pol_cols_needed = [c for c in ["policy_number", "customer_id", "policy_csl"]
                       if c in dim_pol.columns]
    print(f"dim_policy available -- columns used: {pol_cols_needed}")
 
    if "policy_number" in dim_pol.columns:
        dim_pol_slim = (dim_pol
            .select(*pol_cols_needed)
            .withColumnRenamed("policy_number", "policyid")
        )
        claims_raw = claims_raw.join(dim_pol_slim, on="policyid", how="left")
        print("  OK Joined dim_policy -> customer_id and policy_csl added.")
    else:
        raise ValueError("policy_number column missing from dim_policy")
 
except Exception as e:
    print(f"  WARNING dim_policy not available -- skipping: {e}")
    for stub in ["customer_id", "policy_csl"]:
        if stub not in claims_raw.columns:
            claims_raw = claims_raw.withColumn(stub, F.lit(None).cast(T.StringType()))
 
# Step 2: Join to dim_customer via customer_id -> get region
try:
    dim_cust = read_current(f"{SCHEMA_G}.dim_customer")
    print(f"dim_customer available -- columns: {dim_cust.columns}")
 
    if "customer_id" in dim_cust.columns and "region" in dim_cust.columns:
        dim_cust_slim = (dim_cust
            .select("customer_id", F.col("region").alias("customer_region"))
        )
        claims_raw = claims_raw.join(dim_cust_slim, on="customer_id", how="left")
        print("  OK Joined dim_customer -> customer_region added.")
    else:
        raise ValueError("customer_id or region column missing from dim_customer")
 
except Exception as e:
    print(f"  WARNING dim_customer not available -- skipping: {e}")
    if "customer_region" not in claims_raw.columns:
        claims_raw = claims_raw.withColumn("customer_region", F.lit(None).cast(T.StringType()))
 
print(f"\nPost-enrichment columns: {claims_raw.columns}")
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# 1c. Cast numeric columns and derive missing computed fields.
#
# silver_claims actual amount columns: injury, vehicle, property
# We derive:
#   total_claim_amount    = injury + vehicle + property
#   claim_processing_days = datediff(claim_processed_on, claim_logged_on)
#
# Using try_cast / try_to_date throughout because:
#   - Amount fields contain the string literal "NULL" -> hard cast throws
#   - Date fields contain Excel serial fragments like "38:00.0" -> hard parse throws
#   try_cast / try_to_date return NULL on malformed input instead of erroring.
# ------------------------------------------------------------------
 
# Cast three component amount columns
amount_cols = ["injury", "vehicle", "property"]
for nc in amount_cols:
    if nc in claims_raw.columns:
        claims_raw = claims_raw.withColumn(
            nc, F.expr(f"try_cast(`{nc}` as double)")
        )
 
# Cast other numeric fields
other_numeric = ["witnesses", "number_of_vehicles_involved", "bodily_injuries"]
for nc in other_numeric:
    if nc in claims_raw.columns:
        claims_raw = claims_raw.withColumn(
            nc, F.expr(f"try_cast(`{nc}` as double)")
        )
 
# Derive total_claim_amount -- coalesce to 0 so a null part doesn't null the whole sum
claims_raw = claims_raw.withColumn(
    "total_claim_amount",
    F.coalesce(F.col("injury"),   F.lit(0.0)) +
    F.coalesce(F.col("vehicle"),  F.lit(0.0)) +
    F.coalesce(F.col("property"), F.lit(0.0))
)
print("Derived total_claim_amount = injury + vehicle + property")
 
# Derive claim_processing_days from date difference
if "claim_processing_days" not in claims_raw.columns:
    if "claim_logged_on" in claims_raw.columns and "claim_processed_on" in claims_raw.columns:
        for dc in ["claim_logged_on", "claim_processed_on", "incident_date"]:
            if dc in claims_raw.columns:
                claims_raw = claims_raw.withColumn(
                    dc, F.expr(f"try_to_date(`{dc}`, 'yyyy-MM-dd')")
                )
        claims_raw = claims_raw.withColumn(
            "claim_processing_days",
            F.datediff(F.col("claim_processed_on"), F.col("claim_logged_on"))
             .cast(T.DoubleType())
        )
        print("Derived claim_processing_days = claim_processed_on - claim_logged_on")
    else:
        claims_raw = claims_raw.withColumn(
            "claim_processing_days", F.lit(None).cast(T.DoubleType())
        )
        print("claim_processing_days: could not derive -- date columns missing")
else:
    for dc in ["claim_logged_on", "claim_processed_on", "incident_date"]:
        if dc in claims_raw.columns:
            claims_raw = claims_raw.withColumn(
                dc, F.expr(f"try_to_date(`{dc}`, 'yyyy-MM-dd')")
            )
    claims_raw = claims_raw.withColumn(
        "claim_processing_days",
        F.expr("try_cast(claim_processing_days as double)")
    )
 
# Alias component columns to standard names for readability downstream
claims_raw = (claims_raw
    .withColumn("injury_claim",   F.col("injury"))
    .withColumn("vehicle_claim",  F.col("vehicle"))
    .withColumn("property_claim", F.col("property"))
)
 
claims       = claims_raw
total_claims = claims.count()
print(f"\nClaims ready for scoring: {total_claims:,}")
print(f"Sample totals -- max: {claims.agg(F.max('total_claim_amount')).collect()[0][0]:,.0f}  "
      f"mean: {claims.agg(F.mean('total_claim_amount')).collect()[0][0]:,.0f}")

In [0]:
# =============================================================================
# SECTION 2 -- POPULATION STATISTICS
# Computed ONCE on the full population and used as constants in rule scoring.
# =============================================================================
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# 2a. Amount statistics -- used by R1
# ------------------------------------------------------------------
amt_stats = (claims
    .agg(
        F.mean("total_claim_amount").alias("amt_mean"),
        F.stddev_pop("total_claim_amount").alias("amt_std"),
        F.expr("percentile_approx(total_claim_amount, 0.25)").alias("amt_q1"),
        F.expr("percentile_approx(total_claim_amount, 0.75)").alias("amt_q3"),
    )
    .collect()[0]
)
AMT_MEAN = float(amt_stats["amt_mean"] or 0)
AMT_STD  = float(amt_stats["amt_std"]  or 1)
AMT_IQR  = float((amt_stats["amt_q3"] or 0) - (amt_stats["amt_q1"] or 0))
print(f"Amount  -> mean={AMT_MEAN:,.0f}  std={AMT_STD:,.0f}  IQR={AMT_IQR:,.0f}")
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# 2b. Severity-level average amounts -- used by R2
# ------------------------------------------------------------------
sev_stats = (claims
    .filter(F.col("incident_severity").isNotNull())
    .groupBy("incident_severity")
    .agg(
        F.mean("total_claim_amount").alias("sev_mean"),
        F.stddev_pop("total_claim_amount").alias("sev_std"),
    )
)
sev_stats.show(truncate=False)
 
sev_dict = {
    row["incident_severity"]: (
        float(row["sev_mean"] or 0),
        float(row["sev_std"]  or 1),
    )
    for row in sev_stats.collect()
}
print("Severity map:", sev_dict)
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# 2c. Claim counts per customer -- used to populate customer_claim_count
#     in the LLM prompt context (not a scoring rule itself)
# ------------------------------------------------------------------
if "customer_id" in claims.columns and claims.filter(F.col("customer_id").isNotNull()).count() > 0:
    repeat_key = "customer_id"
else:
    repeat_key = "policyid"
    print(f"WARNING: customer_id unavailable -- using {repeat_key} for claim count")

# Drop BEFORE computing repeat_counts to avoid ambiguity at join time
if "customer_claim_count" in claims.columns:
    claims = claims.drop("customer_claim_count")

repeat_counts = (claims
    .groupBy(repeat_key)
    .agg(F.count("claimid").alias("customer_claim_count"))
)

# Use a broadcast hint and alias to guarantee no column ambiguity
claims = claims.join(
    F.broadcast(repeat_counts),
    on=repeat_key,
    how="left"
)

# Coalesce handles any NULLs from the left join
claims = claims.withColumn(
    "customer_claim_count",
    F.coalesce(F.col("customer_claim_count"), F.lit(1))
)

high_freq = claims.filter(F.col("customer_claim_count") > F.lit(3)).select(repeat_key).distinct().count()
print(f"Claim count keyed on : {repeat_key}")
print(f"High-frequency (>3)  : {high_freq}")
 
# COMMAND ----------
 
# =============================================================================
# SECTION 3 -- ANOMALY RULE SCORING
#
# Each rule produces a normalised indicator value in [0, 1].
# 0 = no signal, 1 = maximum signal.
# These feed the CRITIC weight derivation in Section 4.
# =============================================================================
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# R1 -- AMOUNT Z-SCORE
# Why: Inflated claim amounts are the most direct indicator of fraud.
# Method: z-score of total_claim_amount, capped at z=3 -> normalised to [0,1].
# ------------------------------------------------------------------
claims = claims.withColumn(
    "_r1_zscore_raw",
    F.when(
        F.col("total_claim_amount").isNotNull(),
        (F.col("total_claim_amount") - F.lit(AMT_MEAN)) / F.lit(AMT_STD)
    ).otherwise(F.lit(0.0))
)
claims = claims.withColumn(
    "r1_amount_zscore",
    F.least(F.lit(1.0), F.greatest(F.lit(0.0), F.col("_r1_zscore_raw") / F.lit(3.0)))
)
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# R2 -- SEVERITY MISMATCH
# Why: A "Trivial Damage" claim priced like a "Major Damage" claim
#      suggests the severity label was intentionally downgraded to
#      avoid scrutiny while filing a high-value claim.
# Method: within-severity z-score, capped and normalised to [0,1].
# ------------------------------------------------------------------
sev_mean_expr = F.lit(AMT_MEAN)
sev_std_expr  = F.lit(AMT_STD)
for sev, (m, s) in sev_dict.items():
    sev_mean_expr = F.when(F.col("incident_severity") == sev, F.lit(m)).otherwise(sev_mean_expr)
    sev_std_expr  = F.when(F.col("incident_severity") == sev, F.lit(max(s, 1.0))).otherwise(sev_std_expr)
 
claims = claims.withColumn(
    "_r2_sev_z",
    F.when(
        F.col("total_claim_amount").isNotNull(),
        (F.col("total_claim_amount") - sev_mean_expr) / sev_std_expr
    ).otherwise(F.lit(0.0))
)
claims = claims.withColumn(
    "r2_severity_mismatch",
    F.least(F.lit(1.0), F.greatest(F.lit(0.0), F.col("_r2_sev_z") / F.lit(3.0)))
)
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# R3 -- INJURY-TO-VEHICLE RATIO ANOMALY
# Why: Fraudsters inflate injury claims because they are harder to
#      verify than vehicle damage. injury >> vehicle on a minor
#      collision is a textbook auto insurance fraud red flag.
# Method: z-score of injury/(vehicle+1) ratio, capped to [0,1].
#         +1 avoids division-by-zero on zero-vehicle claims.
# ------------------------------------------------------------------
ratio_stats = (claims
    .withColumn("_inj_veh_ratio",
        F.col("injury_claim") / (F.col("vehicle_claim") + F.lit(1.0))
    )
    .agg(
        F.mean("_inj_veh_ratio").alias("ratio_mean"),
        F.stddev_pop("_inj_veh_ratio").alias("ratio_std"),
    )
    .collect()[0]
)
RATIO_MEAN = float(ratio_stats["ratio_mean"] or 0)
RATIO_STD  = float(ratio_stats["ratio_std"]  or 1)
print(f"Injury/Vehicle ratio -> mean={RATIO_MEAN:.3f}  std={RATIO_STD:.3f}")
 
claims = claims.withColumn(
    "_r3_ratio_z",
    F.when(
        F.col("injury_claim").isNotNull() & F.col("vehicle_claim").isNotNull(),
        (
            (F.col("injury_claim") / (F.col("vehicle_claim") + F.lit(1.0)))
            - F.lit(RATIO_MEAN)
        ) / F.lit(max(RATIO_STD, 0.001))
    ).otherwise(F.lit(0.0))
)
claims = claims.withColumn(
    "r3_injury_ratio",
    F.least(F.lit(1.0), F.greatest(F.lit(0.0), F.col("_r3_ratio_z") / F.lit(3.0)))
)
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# R4 -- STAGED ACCIDENT (zero witnesses + no police on multi-vehicle)
# Why: A multi-vehicle collision with zero witnesses and no police
#      report is statistically implausible -- real multi-car accidents
#      almost always involve bystanders and law enforcement.
#      This combination is the hallmark of a staged accident claim.
# Method: 3-condition flag -> score = conditions_present / 3
#   C1 -- number_of_vehicles_involved > 1
#   C2 -- witnesses == 0
#   C3 -- police_report_available != 'YES'
# ------------------------------------------------------------------
has_nv = "number_of_vehicles_involved" in claims.columns
has_wi = "witnesses"                   in claims.columns
has_pr = "police_report_available"     in claims.columns
 
c1 = (F.col("number_of_vehicles_involved") > F.lit(1)).cast(T.DoubleType()) \
     if has_nv else F.lit(0.0)
c2 = (F.col("witnesses") == F.lit(0)).cast(T.DoubleType()) \
     if has_wi else F.lit(0.0)
c3 = (F.upper(F.col("police_report_available")) != F.lit("YES")).cast(T.DoubleType()) \
     if has_pr else F.lit(0.0)
 
claims = claims.withColumn(
    "r4_staged_accident",
    F.round(
        (F.coalesce(c1, F.lit(0.0)) +
         F.coalesce(c2, F.lit(0.0)) +
         F.coalesce(c3, F.lit(0.0))) / F.lit(3.0),
        4
    )
)
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# R5 -- DOCUMENTATION GAP (3-signal)
# Why: Missing documentation is the hallmark of phantom/staged claims.
#      Real accidents generate a paper trail. We check three signals:
#        S1 -- property_damage is NULL       (cleaned from "?" in Silver)
#        S2 -- police_report_available is NULL or not YES
#        S3 -- authorities_contacted == "None"
#      Score = signals_present / 3 -> 0 / 0.33 / 0.67 / 1.0
# ------------------------------------------------------------------
has_pd   = "property_damage"         in claims.columns
has_pr   = "police_report_available" in claims.columns
has_auth = "authorities_contacted"   in claims.columns
 
s1 = F.col("property_damage").isNull().cast(T.DoubleType()) \
     if has_pd else F.lit(0.0)
 
s2 = (
    F.col("police_report_available").isNull() |
    (F.upper(F.col("police_report_available")) != F.lit("YES"))
).cast(T.DoubleType()) if has_pr else F.lit(0.0)
 
s3 = (F.lower(F.col("authorities_contacted")) == F.lit("none")).cast(T.DoubleType()) \
     if has_auth else F.lit(0.0)
 
claims = claims.withColumn(
    "r5_doc_gap",
    F.round(
        (F.coalesce(s1, F.lit(0.0)) +
         F.coalesce(s2, F.lit(0.0)) +
         F.coalesce(s3, F.lit(0.0))) / F.lit(3.0),
        4
    )
)
 

In [0]:
# =============================================================================
# SECTION 3 -- ANOMALY RULE SCORING
#
# Each rule produces a normalised indicator value in [0, 1].
# 0 = no signal, 1 = maximum signal.
# These feed the CRITIC weight derivation in Section 4.
# =============================================================================
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# R1 -- AMOUNT Z-SCORE
# Why: Inflated claim amounts are the most direct indicator of fraud.
# Method: z-score of total_claim_amount, capped at z=3 -> normalised to [0,1].
# ------------------------------------------------------------------
claims = claims.withColumn(
    "_r1_zscore_raw",
    F.when(
        F.col("total_claim_amount").isNotNull(),
        (F.col("total_claim_amount") - F.lit(AMT_MEAN)) / F.lit(AMT_STD)
    ).otherwise(F.lit(0.0))
)
claims = claims.withColumn(
    "r1_amount_zscore",
    F.least(F.lit(1.0), F.greatest(F.lit(0.0), F.col("_r1_zscore_raw") / F.lit(3.0)))
)
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# R2 -- SEVERITY MISMATCH
# Why: A "Trivial Damage" claim priced like a "Major Damage" claim
#      suggests the severity label was intentionally downgraded to
#      avoid scrutiny while filing a high-value claim.
# Method: within-severity z-score, capped and normalised to [0,1].
# ------------------------------------------------------------------
sev_mean_expr = F.lit(AMT_MEAN)
sev_std_expr  = F.lit(AMT_STD)
for sev, (m, s) in sev_dict.items():
    sev_mean_expr = F.when(F.col("incident_severity") == sev, F.lit(m)).otherwise(sev_mean_expr)
    sev_std_expr  = F.when(F.col("incident_severity") == sev, F.lit(max(s, 1.0))).otherwise(sev_std_expr)
 
claims = claims.withColumn(
    "_r2_sev_z",
    F.when(
        F.col("total_claim_amount").isNotNull(),
        (F.col("total_claim_amount") - sev_mean_expr) / sev_std_expr
    ).otherwise(F.lit(0.0))
)
claims = claims.withColumn(
    "r2_severity_mismatch",
    F.least(F.lit(1.0), F.greatest(F.lit(0.0), F.col("_r2_sev_z") / F.lit(3.0)))
)
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# R3 -- INJURY-TO-VEHICLE RATIO ANOMALY
# Why: Fraudsters inflate injury claims because they are harder to
#      verify than vehicle damage. injury >> vehicle on a minor
#      collision is a textbook auto insurance fraud red flag.
# Method: z-score of injury/(vehicle+1) ratio, capped to [0,1].
#         +1 avoids division-by-zero on zero-vehicle claims.
# ------------------------------------------------------------------
ratio_stats = (claims
    .withColumn("_inj_veh_ratio",
        F.col("injury_claim") / (F.col("vehicle_claim") + F.lit(1.0))
    )
    .agg(
        F.mean("_inj_veh_ratio").alias("ratio_mean"),
        F.stddev_pop("_inj_veh_ratio").alias("ratio_std"),
    )
    .collect()[0]
)
RATIO_MEAN = float(ratio_stats["ratio_mean"] or 0)
RATIO_STD  = float(ratio_stats["ratio_std"]  or 1)
print(f"Injury/Vehicle ratio -> mean={RATIO_MEAN:.3f}  std={RATIO_STD:.3f}")
 
claims = claims.withColumn(
    "_r3_ratio_z",
    F.when(
        F.col("injury_claim").isNotNull() & F.col("vehicle_claim").isNotNull(),
        (
            (F.col("injury_claim") / (F.col("vehicle_claim") + F.lit(1.0)))
            - F.lit(RATIO_MEAN)
        ) / F.lit(max(RATIO_STD, 0.001))
    ).otherwise(F.lit(0.0))
)
claims = claims.withColumn(
    "r3_injury_ratio",
    F.least(F.lit(1.0), F.greatest(F.lit(0.0), F.col("_r3_ratio_z") / F.lit(3.0)))
)
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# R4 -- STAGED ACCIDENT (zero witnesses + no police on multi-vehicle)
# Why: A multi-vehicle collision with zero witnesses and no police
#      report is statistically implausible -- real multi-car accidents
#      almost always involve bystanders and law enforcement.
#      This combination is the hallmark of a staged accident claim.
# Method: 3-condition flag -> score = conditions_present / 3
#   C1 -- number_of_vehicles_involved > 1
#   C2 -- witnesses == 0
#   C3 -- police_report_available != 'YES'
# ------------------------------------------------------------------
has_nv = "number_of_vehicles_involved" in claims.columns
has_wi = "witnesses"                   in claims.columns
has_pr = "police_report_available"     in claims.columns
 
c1 = (F.col("number_of_vehicles_involved") > F.lit(1)).cast(T.DoubleType()) \
     if has_nv else F.lit(0.0)
c2 = (F.col("witnesses") == F.lit(0)).cast(T.DoubleType()) \
     if has_wi else F.lit(0.0)
c3 = (F.upper(F.col("police_report_available")) != F.lit("YES")).cast(T.DoubleType()) \
     if has_pr else F.lit(0.0)
 
claims = claims.withColumn(
    "r4_staged_accident",
    F.round(
        (F.coalesce(c1, F.lit(0.0)) +
         F.coalesce(c2, F.lit(0.0)) +
         F.coalesce(c3, F.lit(0.0))) / F.lit(3.0),
        4
    )
)
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# R5 -- DOCUMENTATION GAP (3-signal)
# Why: Missing documentation is the hallmark of phantom/staged claims.
#      Real accidents generate a paper trail. We check three signals:
#        S1 -- property_damage is NULL       (cleaned from "?" in Silver)
#        S2 -- police_report_available is NULL or not YES
#        S3 -- authorities_contacted == "None"
#      Score = signals_present / 3 -> 0 / 0.33 / 0.67 / 1.0
# ------------------------------------------------------------------
has_pd   = "property_damage"         in claims.columns
has_pr   = "police_report_available" in claims.columns
has_auth = "authorities_contacted"   in claims.columns
 
s1 = F.col("property_damage").isNull().cast(T.DoubleType()) \
     if has_pd else F.lit(0.0)
 
s2 = (
    F.col("police_report_available").isNull() |
    (F.upper(F.col("police_report_available")) != F.lit("YES"))
).cast(T.DoubleType()) if has_pr else F.lit(0.0)
 
s3 = (F.lower(F.col("authorities_contacted")) == F.lit("none")).cast(T.DoubleType()) \
     if has_auth else F.lit(0.0)
 
claims = claims.withColumn(
    "r5_doc_gap",
    F.round(
        (F.coalesce(s1, F.lit(0.0)) +
         F.coalesce(s2, F.lit(0.0)) +
         F.coalesce(s3, F.lit(0.0))) / F.lit(3.0),
        4
    )
)
 

In [0]:

# =============================================================================
# SECTION 4 -- CRITIC WEIGHT DERIVATION
#
# CRITIC = CRiteria Importance Through Intercriteria Correlation
# Source: Diakoulaki, Mavrotas & Papayannakis (1995), Computers & Operations Research
#
# WHY CRITIC?
#   Fixed weights are arbitrary and indefensible under audit.
#   CRITIC derives weights from the data:
#     C_j = sigma_j x sum_k (1 - r_jk)
#   A rule with HIGH variance and LOW correlation with others -> high weight.
#   Weights are normalised so they sum to 1.
# =============================================================================
 
# COMMAND ----------
 
INDICATOR_COLS = [
    "r1_amount_zscore",
    "r2_severity_mismatch",
    "r3_injury_ratio",
    "r4_staged_accident",
    "r5_doc_gap",
]
 
# Collect indicator matrix to driver (5 cols x ~3000 rows -- safe)
indicator_pdf = (claims
    .select(*INDICATOR_COLS)
    .fillna(0)
    .toPandas()
)
 
# CRITIC Step 1: population std-dev per indicator
sigma = indicator_pdf.std(ddof=0).values
sigma = np.where(sigma < 1e-9, 1e-9, sigma)   # guard zero-variance
 
# CRITIC Step 2: Pearson correlation matrix
corr_matrix = indicator_pdf.corr(method="pearson").values
corr_matrix = np.nan_to_num(corr_matrix, nan=0.0)
 
# CRITIC Step 3: information content  C_j = sigma_j x sum_k (1 - r_jk)
conflict = np.sum(1 - corr_matrix, axis=1)
C        = sigma * conflict
 
# CRITIC Step 4: normalise -> weights sum to 1
W = C / C.sum()
 
print("\n=== CRITIC WEIGHT DERIVATION ===")
print(f"{'Rule':<25} {'sigma':>8} {'Conflict':>10} {'C_j':>10} {'Weight':>8}")
print("-" * 65)
for i, col in enumerate(INDICATOR_COLS):
    print(f"{col:<25} {sigma[i]:>8.4f} {conflict[i]:>10.4f} {C[i]:>10.4f} {W[i]:>8.4f}")
print(f"\nWeights sum to: {W.sum():.6f}")
 
w1, w2, w3, w4, w5 = [float(x) for x in W]
 
 
# COMMAND ----------

In [0]:
# =============================================================================
# SECTION 5 -- COMPOSITE ANOMALY SCORE
#
# score = 100 x sum_j (w_j x indicator_j)   in [0, 100]
#
# THRESHOLD RATIONALE (40/100):
#   Under equal weights (0.20 each), score=40 requires at least two
#   rules at full signal, or one strong + multiple partial -- a meaningful
#   multi-dimensional pattern, not single-rule noise.
# =============================================================================
 
# COMMAND ----------
 
claims = claims.withColumn(
    "anomaly_score",
    F.round(
        F.lit(100.0) * (
            F.lit(w1) * F.col("r1_amount_zscore")    +
            F.lit(w2) * F.col("r2_severity_mismatch") +
            F.lit(w3) * F.col("r3_injury_ratio")      +
            F.lit(w4) * F.col("r4_staged_accident")   +
            F.lit(w5) * F.col("r5_doc_gap")
        ),
        2
    )
)
 
# Priority tier
claims = claims.withColumn(
    "priority_tier",
    F.when(F.col("anomaly_score") >= F.lit(HIGH_THRESHOLD),    F.lit("HIGH"))
     .when(F.col("anomaly_score") >= F.lit(ANOMALY_THRESHOLD), F.lit("MEDIUM"))
     .otherwise(F.lit("LOW"))
)
 
# Human-readable triggered rules string
claims = claims.withColumn(
    "triggered_rules",
    F.concat_ws(", ",
        F.when(F.col("r1_amount_zscore")    >= 0.4, F.lit("R1:HighAmount")).otherwise(F.lit(None)),
        F.when(F.col("r2_severity_mismatch") >= 0.4, F.lit("R2:SeverityMismatch")).otherwise(F.lit(None)),
        F.when(F.col("r3_injury_ratio")      >= 0.4, F.lit("R3:InjuryRatio")).otherwise(F.lit(None)),
        F.when(F.col("r4_staged_accident")   >= 0.4, F.lit("R4:StagedAccident")).otherwise(F.lit(None)),
        F.when(F.col("r5_doc_gap")           >= 0.3, F.lit("R5:DocGap")).otherwise(F.lit(None)),
    )
)
 
# Summary
claims.groupBy("priority_tier").count().orderBy("priority_tier").show()
flagged_count = claims.filter(F.col("priority_tier").isin(["HIGH", "MEDIUM"])).count()
print(f"Total: {total_claims:,}  |  Flagged: {flagged_count:,}  ({100*flagged_count/total_claims:.1f}%)")

In [0]:
# =============================================================================
# SECTION 6 -- AI INVESTIGATION BRIEF GENERATION
#
# LLM is used ONLY as an explainer -- it narrates what statistics detected.
# AI briefs are generated for the TOP 10 highest-scoring flagged claims only.
# All other flagged claims land in the output table with NULL brief.
# =============================================================================
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# 6a. Collect TOP 10 flagged claims to driver for LLM calls
# ------------------------------------------------------------------
LLM_BRIEF_LIMIT = 3
BRIEF_COLS = [
    "claimid", "customer_id", "policyid",
    "incident_date", "incident_severity", "incident_type",
    "total_claim_amount", "vehicle_claim", "property_claim", "injury_claim",
    "claim_processing_days",
    "property_damage", "police_report_available", "authorities_contacted",
    "number_of_vehicles_involved", "witnesses",
    "claim_rejected", "customer_claim_count",
    "anomaly_score", "priority_tier", "triggered_rules",
    "r1_amount_zscore", "r2_severity_mismatch", "r3_injury_ratio",
    "r4_staged_accident", "r5_doc_gap",
    "customer_region", "policy_csl",
]
brief_cols_available = [c for c in BRIEF_COLS if c in claims.columns]
 
# Only top LLM_BRIEF_LIMIT claims by score get a brief
flagged_pdf = (claims
    .filter(F.col("priority_tier").isin(["HIGH", "MEDIUM"]))
    .orderBy(F.desc("anomaly_score"))
    .limit(LLM_BRIEF_LIMIT)
    .select(*brief_cols_available)
    .fillna({"property_damage": "Unknown", "police_report_available": "Unknown"})
    .toPandas()
)
print(f"Claims selected for AI brief generation: {len(flagged_pdf)} (top {LLM_BRIEF_LIMIT} by score)")
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# 6b. Prompt builder
# ------------------------------------------------------------------
def build_prompt(row: dict) -> str:
    """
    Structured, data-grounded prompt. Passes specific numeric indicators
    so the LLM narrates what was already computed, not hallucinations.
    """
    rules_fired = row.get("triggered_rules", "None")
    r1    = float(row.get("r1_amount_zscore",    0) or 0)
    r2    = float(row.get("r2_severity_mismatch", 0) or 0)
    r3    = float(row.get("r3_injury_ratio",      0) or 0)
    r4    = float(row.get("r4_staged_accident",   0) or 0)
    r5    = float(row.get("r5_doc_gap",           0) or 0)
    score = float(row.get("anomaly_score",        0) or 0)
 
    return f"""You are a senior fraud investigation analyst at PrimeInsurance.
A statistical anomaly engine has flagged claim {row.get('claimid', 'UNKNOWN')} for investigation.
Your job is to write a concise, structured investigation brief that explains the findings.
 
=== CLAIM DATA ===
Claim ID          : {row.get('claimid')}
Customer ID       : {row.get('customer_id')}
Policy ID         : {row.get('policyid')}
Incident Date     : {row.get('incident_date')}
Incident Severity : {row.get('incident_severity')}
Incident Type     : {row.get('incident_type')}
Total Claim Amount: ${row.get('total_claim_amount', 0):,.2f}
  - Vehicle Claim : ${row.get('vehicle_claim', 0):,.2f}
  - Property Claim: ${row.get('property_claim', 0):,.2f}
  - Injury Claim  : ${row.get('injury_claim', 0):,.2f}
Processing Days   : {row.get('claim_processing_days')}
Vehicles Involved : {row.get('number_of_vehicles_involved')}
Witnesses         : {row.get('witnesses')}
Authorities       : {row.get('authorities_contacted')}
Property Damage   : {row.get('property_damage')}
Police Report     : {row.get('police_report_available')}
Claim Rejected    : {row.get('claim_rejected')}
Prior Claims      : {row.get('customer_claim_count')}
Region            : {row.get('customer_region')}
Policy CSL        : {row.get('policy_csl')}
 
=== ANOMALY SCORE BREAKDOWN ===
Overall Anomaly Score : {score:.1f} / 100  (threshold = {ANOMALY_THRESHOLD})
Priority Tier         : {row.get('priority_tier')}
Rules Triggered       : {rules_fired}
 
Rule Indicator Values (0=no signal, 1=maximum signal):
  R1 - Amount Z-Score       : {r1:.3f}  (claim amount vs population)
  R2 - Severity Mismatch    : {r2:.3f}  (amount vs severity-peer group)
  R3 - Injury/Vehicle Ratio : {r3:.3f}  (injury disproportionate to vehicle damage)
  R4 - Staged Accident      : {r4:.3f}  (multi-vehicle, zero witnesses, no police report)
  R5 - Documentation Gap    : {r5:.3f}  (missing property damage, police report, authority contact)
 
=== INSTRUCTIONS ===
Write a professional investigation brief with the following three sections.
Use the SPECIFIC DATA POINTS above -- do not make generic statements.
 
**SECTION 1 -- WHY THIS CLAIM IS SUSPICIOUS**
Explain which specific data points triggered the rules and why each is unusual.
Reference actual numbers (e.g. "$X,XXX claim amount", "X witnesses", "X processing days").
 
**SECTION 2 -- RISK FACTORS PRESENT**
List each triggered rule and explain what fraud risk it represents in the context of
auto insurance investigation. Keep this factual and grounded in the data.
 
**SECTION 3 -- RECOMMENDED INVESTIGATOR ACTIONS**
Provide 3-5 specific, actionable next steps.
Be concrete: name the document to request, the party to contact, the system to query.
 
Keep the entire brief under 400 words. Never say "the claimant is guilty" or use
speculative language. Use professional language appropriate for a regulated environment.
"""
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# 6c. LLM call -- with timeout and retry
# ------------------------------------------------------------------
def call_llm(prompt: str, retries: int = 2, timeout: int = 30) -> str:
    """
    OpenAI-compatible call -- works for both TEST_MODE (OpenRouter)
    and PRODUCTION (Databricks Foundation Model).
    timeout : max seconds per attempt
    retries : number of retry attempts on failure
    """
    for attempt in range(1, retries + 2):
        try:
            response = client.chat.completions.create(
                model      = MODEL_ID,
                messages   = [{"role": "user", "content": prompt}],
                max_tokens = 1024,
                timeout    = timeout,
            )
            content = response.choices[0].message.content
            # Databricks returns content as a list of dicts; OpenRouter returns a string
            if isinstance(content, list):
                return " ".join(
                    block.get("text", "") for block in content
                    if isinstance(block, dict)
                ).strip()
            return content.strip()
        except Exception as e:
            err = str(e)[:120]
            if attempt <= retries:
                print(f"    WARNING Attempt {attempt} failed ({err}) -- retrying...")
            else:
                return f"[LLM_ERROR after {retries+1} attempts: {err}]"
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# 6d. Generate briefs -- per-claim progress print
# ------------------------------------------------------------------
briefs       = {}
total_to_gen = len(flagged_pdf)
print(f"Generating {total_to_gen} AI briefs (top {LLM_BRIEF_LIMIT} by anomaly score)...\n")
 
for i, (_, row) in enumerate(flagged_pdf.iterrows()):
    claim_id = str(row.get("claimid", f"row_{i}"))
    tier     = row.get("priority_tier", "?")
    score    = row.get("anomaly_score", 0)
 
    print(f"  [{i+1:>2}/{total_to_gen}]  claim={claim_id}  tier={tier}  score={score} ...", end=" ")
 
    brief            = call_llm(build_prompt(row.to_dict()))
    briefs[claim_id] = brief
 
    preview = brief[:60].replace("\n", " ")
    print(f"OK  '{preview}...'")
 
error_count = sum(1 for b in briefs.values() if b.startswith("[LLM_ERROR"))
print(f"\nBriefs generated: {len(briefs)}  |  Errors: {error_count}")

In [0]:

# =============================================================================
# SECTION 7 -- ASSEMBLE OUTPUT TABLES
#
# TABLE A -- primeins.gold.claim_anomaly_scores
#   Every claim, scored and tiered. No AI brief. For dashboards and audit.
#
# TABLE B -- primeins.gold.claim_anomaly_explanations
#   Flagged claims only (HIGH + MEDIUM). ai_investigation_brief populated
#   for top 10 by score; NULL for remaining flagged claims.
# =============================================================================
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# 7a. TABLE A -- All claims with anomaly scores
# ------------------------------------------------------------------
SCORE_COLS = [
    "claimid", "customer_id", "policyid",
    "incident_date", "incident_severity", "incident_type",
    "total_claim_amount", "vehicle_claim", "property_claim", "injury_claim",
    "claim_processing_days", "property_damage", "police_report_available",
    "authorities_contacted", "number_of_vehicles_involved", "witnesses",
    "claim_rejected", "customer_claim_count",
    "customer_region", "policy_csl",
    "r1_amount_zscore", "r2_severity_mismatch", "r3_injury_ratio",
    "r4_staged_accident", "r5_doc_gap",
    "anomaly_score", "priority_tier", "triggered_rules",
]
score_cols_available = [c for c in SCORE_COLS if c in claims.columns]
 
all_claims_scored = (
    claims
    .select(*score_cols_available)
    .withColumn("is_flagged",      F.col("priority_tier").isin(["HIGH", "MEDIUM"]))
    .withColumn("_anomaly_run_id", F.lit(RUN_ID))
    .withColumn("_gold_run_id",    F.lit(RUN_ID))
    .withColumn("_gold_load_ts",   F.current_timestamp())
    .withColumn("_stage",          F.lit("claim_anomaly_scores"))
)
 
write_gold(all_claims_scored, "claim_anomaly_scores",
           f"ALL claims | run={RUN_ID} | flagged={flagged_count:,}/{total_claims:,}")
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# 7b. TABLE B -- Flagged claims with AI briefs (top 10 only)
#     All flagged claims land here.
#     ai_investigation_brief is NULL for those outside the top 10.
# ------------------------------------------------------------------
brief_rows = [(str(cid), txt) for cid, txt in briefs.items()]
briefs_df  = spark.createDataFrame(brief_rows, schema=T.StructType([
    T.StructField("claimid",                T.StringType(), True),
    T.StructField("ai_investigation_brief", T.StringType(), True),
]))
 
flagged_with_briefs = (
    all_claims_scored
    .filter(F.col("is_flagged") == True)
    .join(briefs_df, on="claimid", how="left")
    # NULL = flagged but outside top 10 -- awaiting investigation, not an error
    .withColumn("_stage", F.lit("claim_anomaly_explanations"))
)
 
write_gold(flagged_with_briefs, "claim_anomaly_explanations",
           f"FLAGGED claims | top {LLM_BRIEF_LIMIT} have AI briefs | mode={'TEST' if TEST_MODE else 'PROD'} | model={MODEL_ID} | run={RUN_ID}")

In [0]:
# =============================================================================
# SECTION 8 -- VERIFICATION & SAMPLE OUTPUT
# =============================================================================
 
# COMMAND ----------
 
print("=== SCHEMA: claim_anomaly_scores ===")
spark.table(f"{SCHEMA_G}.claim_anomaly_scores").printSchema()
 
print("=== SCHEMA: claim_anomaly_explanations ===")
spark.table(f"{SCHEMA_G}.claim_anomaly_explanations").printSchema()
 
# COMMAND ----------
 
print("=== PRIORITY TIER DISTRIBUTION ===")
(spark.table(f"{SCHEMA_G}.claim_anomaly_scores")
    .groupBy("priority_tier")
    .agg(
        F.count("*").alias("claim_count"),
        F.round(F.avg("anomaly_score"), 2).alias("avg_score"),
        F.round(F.avg("total_claim_amount"), 2).alias("avg_claim_amount"),
    )
    .orderBy(F.desc("avg_score"))
    .show()
)
 
# COMMAND ----------
 
print("=== TOP 5 HIGH PRIORITY CLAIMS ===")
(spark.table(f"{SCHEMA_G}.claim_anomaly_explanations")
    .filter(F.col("priority_tier") == "HIGH")
    .orderBy(F.desc("anomaly_score"))
    .select("claimid", "anomaly_score", "triggered_rules",
            "total_claim_amount", "incident_severity",
            "number_of_vehicles_involved", "witnesses")
    .limit(5)
    .show(truncate=False)
)
 
# COMMAND ----------
 
print("=== SAMPLE AI INVESTIGATION BRIEFS (top 3 with brief) ===\n")
samples = (spark.table(f"{SCHEMA_G}.claim_anomaly_explanations")
    .filter(
        (F.col("priority_tier") == "HIGH") &
        F.col("ai_investigation_brief").isNotNull()
    )
    .orderBy(F.desc("anomaly_score"))
    .select("claimid", "anomaly_score", "priority_tier",
            "triggered_rules", "ai_investigation_brief")
    .limit(3)
    .collect()
)
for row in samples:
    print("=" * 80)
    print(f"CLAIM ID    : {row['claimid']}")
    print(f"SCORE       : {row['anomaly_score']} / 100")
    print(f"TIER        : {row['priority_tier']}")
    print(f"RULES FIRED : {row['triggered_rules']}")
    print("-" * 80)
    print(row["ai_investigation_brief"])
    print()
 

In [0]:
# =============================================================================
# SECTION 9 -- WEIGHT AUDIT TABLE
# CRITIC-derived weights stored per run for regulatory traceability.
# =============================================================================
 
# COMMAND ----------
 
weight_rows = [
    (col, float(sigma[i]), float(conflict[i]), float(C[i]), float(W[i]), RUN_ID)
    for i, col in enumerate(INDICATOR_COLS)
]
weight_schema = T.StructType([
    T.StructField("rule_name",           T.StringType(), True),
    T.StructField("std_dev",             T.DoubleType(), True),
    T.StructField("conflict_score",      T.DoubleType(), True),
    T.StructField("information_content", T.DoubleType(), True),
    T.StructField("critic_weight",       T.DoubleType(), True),
    T.StructField("_run_id",             T.StringType(), True),
])
weight_df = spark.createDataFrame(weight_rows, schema=weight_schema)
 
write_gold(weight_df, "claim_anomaly_weights",
           f"CRITIC weight audit | run={RUN_ID}")
 
spark.table(f"{SCHEMA_G}.claim_anomaly_weights").show(truncate=False)
 
# COMMAND ----------
 
print("\n" + "=" * 60)
print("CLAIMS ANOMALY ENGINE COMPLETE")
print(f"  Mode             : {'TEST (OpenRouter)' if TEST_MODE else 'PRODUCTION (Databricks)'}")
print(f"  Model            : {MODEL_ID}")
print(f"  Run ID           : {RUN_ID}")
print(f"  Total claims     : {total_claims:,}")
print(f"  Flagged claims   : {flagged_count:,}  ({100*flagged_count/total_claims:.1f}%)")
print(f"  AI briefs written: {len(briefs)}  (top {LLM_BRIEF_LIMIT} by score)")
print(f"  Scores table     : {SCHEMA_G}.claim_anomaly_scores")
print(f"  Briefs table     : {SCHEMA_G}.claim_anomaly_explanations")
print(f"  Weight audit     : {SCHEMA_G}.claim_anomaly_weights")
print("=" * 60)